# 03 — Error analysis

This notebook inspects the held-out test errors from the fixed 70/15/15 split and the six `other` holdout documents. It uses the trained bundle and `src.predict.predict_text`, so the routing behavior matches the API contract.

In [1]:
import sys
from pathlib import Path

import joblib
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import load_dataset
from src.predict import predict_text
from src.train import stratified_split

DATA_DIR = ROOT.parent / "trellis_assessment_ds"
REPORTS_DIR = ROOT / "reports"
bundle = joblib.load(ROOT / "models" / "document_classifier.joblib")
dataset = load_dataset(DATA_DIR)
splits = stratified_split(
    dataset.known_texts, dataset.known_labels, dataset.known_paths,
    val_size=0.15, test_size=0.15, random_state=42,
)
len(splits["test"]["texts"]), len(dataset.other_texts), bundle["confidence_thresholds"]

(149, 6, {'auto_accept': 0.9, 'manual_review': 0.7, 'other': np.float64(0.15)})

## Confusion matrix

The final model has only one raw-label error on the held-out test set: a `space` document predicted as `graphics`. The confidence is low enough that the threshold policy routes it to `other`, which is conservative but also illustrates the known recall trade-off.

In [2]:
pd.read_csv(REPORTS_DIR / "confusion_matrix.csv", index_col="actual")

,business,entertainment,food,graphics,historical,medical,politics,space,sport,technology
actual,,,,,,,,,,
business,15,0,0,0,0,0,0,0,0,0
entertainment,0,15,0,0,0,0,0,0,0,0
food,0,0,15,0,0,0,0,0,0,0
graphics,0,0,0,15,0,0,0,0,0,0
historical,0,0,0,0,15,0,0,0,0,0
medical,0,0,0,0,0,15,0,0,0,0
politics,0,0,0,0,0,0,14,0,0,0
space,0,0,0,1,0,0,0,14,0,0
sport,0,0,0,0,0,0,0,0,15,0


## Misclassified examples

`reports/misclassified_examples.csv` stores the complete row with path, labels, confidence, decision, top-k probabilities, and an excerpt.

In [3]:
misclassified = pd.read_csv(REPORTS_DIR / "misclassified_examples.csv")
misclassified[["path", "actual_label", "raw_label", "final_label", "confidence", "decision"]]

,path,actual_label,raw_label,final_label,confidence,decision
0,trellis_assessment_ds/space/space_18.txt,space,graphics,other,0.1475,fallback_other


## Confidence bands

Most correct validation predictions are below 0.5 confidence, so the fallback threshold must stay low. That is why the selected `other` threshold is 0.15 rather than the initial default of 0.55.

In [4]:
pd.read_csv(REPORTS_DIR / "confidence_ranges.csv")

,confidence_range,total,correct,accuracy
0,"[0.0, 0.5)",118,114,0.9661
1,"[0.5, 0.6)",18,18,1.0000
2,"[0.6, 0.7)",11,11,1.0000
3,"[0.7, 0.8)",1,1,1.0000
4,"[0.8, 0.9)",1,1,1.0000
5,"[0.9, 1.0]",0,0,NaN


## `other` holdout

The six OOD files are all routed to `other` at threshold 0.15. This is useful evidence but still anecdotal because six files are not enough to estimate OOD recall precisely.

In [5]:
other_predictions = pd.read_csv(REPORTS_DIR / "other_holdout_predictions.csv")
other_predictions[["path", "raw_label", "final_label", "confidence", "decision"]]

,path,raw_label,final_label,confidence,decision
0,trellis_assessment_ds/other/other_1.txt,entertainment,other,0.1397,fallback_other
1,trellis_assessment_ds/other/other_2.txt,sport,other,0.1301,fallback_other
2,trellis_assessment_ds/other/other_3.txt,politics,other,0.1176,fallback_other
3,trellis_assessment_ds/other/other_4.txt,graphics,other,0.1326,fallback_other
4,trellis_assessment_ds/other/other_5.txt,technology,other,0.1317,fallback_other
5,trellis_assessment_ds/other/other_6.txt,graphics,other,0.1285,fallback_other


## Takeaways

- Raw held-out accuracy is high (99.33%), with one `space`/`graphics` confusion.
- Probabilities are intentionally treated as routing signals, not just accuracy scores. Many correct known-class predictions are low confidence.
- The selected threshold catches all six OOD examples while misrouting 3/149 validation known docs and 2/149 test known docs.
- The OOD result should be revisited with a larger and more representative negative set before production rollout.